# Sampling Activations from PredNet

Extracts ConvLSTM hidden-state activations from PredNet (Lotter et al., ICLR 2017), a predictive coding model of visual cortex. Each of 4 levels contains a ConvLSTM whose hidden state `R` evolves frame-by-frame — this is true temporal recurrence, analogous to FNN's recurrent core.

The PredNet code is embedded inline below (adapted from https://github.com/leido/pytorch-prednet), patched for CPU/device-agnostic use and extended with a `forward_capture` method that records `R` at each timestep.

**Pretrained weights**: no PyTorch weights are publicly available for PredNet. This notebook uses **randomly-initialized** weights, which still demonstrate the temporal dynamics of the ConvLSTM architecture.

Output: `data/sampled/tensor4d_prednet_L<i>_i3_n<N>_seed17.npy` (shape: neurons × 11 × 8 × 37) for levels L0–L3.

In [7]:
########## INLINE PREDNET IMPLEMENTATION (patched from leido/pytorch-prednet) ##########

import torch
import math
import torch.nn as nn
from torch.nn import Parameter
from torch.nn import functional as F
from torch.nn.modules.utils import _pair


class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=1, dilation=1, groups=1, bias=True):
        super().__init__()
        kernel_size = _pair(kernel_size)
        stride = _pair(stride)
        padding = _pair(padding)
        dilation = _pair(dilation)
        self.out_channels = out_channels
        self.padding_h = tuple(k // 2 for k, s, p, d in zip(kernel_size, stride, padding, dilation))
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.groups = groups
        self.weight_ih = Parameter(torch.Tensor(4 * out_channels, in_channels // groups, *kernel_size))
        self.weight_hh = Parameter(torch.Tensor(4 * out_channels, out_channels // groups, *kernel_size))
        self.weight_ch = Parameter(torch.Tensor(3 * out_channels, out_channels // groups, *kernel_size))
        self.bias_ih = Parameter(torch.Tensor(4 * out_channels))
        self.bias_hh = Parameter(torch.Tensor(4 * out_channels))
        self.bias_ch = Parameter(torch.Tensor(3 * out_channels))
        self.reset_parameters()

    def reset_parameters(self):
        n = 4 * self.in_channels if hasattr(self, 'in_channels') else 4
        stdv = 1. / math.sqrt(max(n, 1))
        for p in self.parameters():
            p.data.uniform_(-stdv, stdv)

    def forward(self, input, hx):
        h_0, c_0 = hx
        wx = F.conv2d(input, self.weight_ih, self.bias_ih,
                      self.stride, self.padding, self.dilation, self.groups)
        wh = F.conv2d(h_0, self.weight_hh, self.bias_hh,
                      self.stride, self.padding_h, self.dilation, self.groups)
        wc = F.conv2d(c_0, self.weight_ch, self.bias_ch,
                      self.stride, self.padding_h, self.dilation, self.groups)

        # Peephole connections: i, f gates see c_0; o gate not wired to c_0 here
        wc_blank = torch.zeros(wc.size(0), self.out_channels, wc.size(2), wc.size(3),
                               device=input.device)
        wxhc = wx + wh + torch.cat([wc[:, :2 * self.out_channels],
                                     wc_blank,
                                     wc[:, 2 * self.out_channels:]], 1)

        i = torch.sigmoid(wxhc[:, :self.out_channels])
        f = torch.sigmoid(wxhc[:, self.out_channels:2 * self.out_channels])
        g = torch.tanh(wxhc[:, 2 * self.out_channels:3 * self.out_channels])
        o = torch.sigmoid(wxhc[:, 3 * self.out_channels:])

        c_1 = f * c_0 + i * g
        h_1 = o * torch.tanh(c_1)
        return h_1, (h_1, c_1)


class SatLU(nn.Module):
    def forward(self, x):
        return x.clamp(0, 1)


class PredNet(nn.Module):
    def __init__(self, R_channels, A_channels):
        super().__init__()
        self.r_channels = R_channels + (0,)  # padding for convenience
        self.a_channels = A_channels
        self.n_layers = len(R_channels)

        for i in range(self.n_layers):
            cell = ConvLSTMCell(
                2 * self.a_channels[i] + self.r_channels[i + 1],
                self.r_channels[i], (3, 3))
            setattr(self, f'cell{i}', cell)

        for i in range(self.n_layers):
            conv = nn.Sequential(
                nn.Conv2d(self.r_channels[i], self.a_channels[i], 3, padding=1),
                nn.ReLU())
            if i == 0:
                conv.add_module('satlu', SatLU())
            setattr(self, f'conv{i}', conv)

        self.upsample = nn.Upsample(scale_factor=2)
        self.maxpool  = nn.MaxPool2d(kernel_size=2, stride=2)

        for l in range(self.n_layers - 1):
            update_A = nn.Sequential(
                nn.Conv2d(2 * self.a_channels[l], self.a_channels[l + 1], 3, padding=1),
                self.maxpool)
            setattr(self, f'update_A{l}', update_A)

    def forward_capture(self, input_seq):
        """Run PredNet frame-by-frame, capturing ConvLSTM hidden state R per level.

        Args:
            input_seq: (B, T, C, H, W) tensor

        Returns:
            all_R: list of length n_layers, each tensor (B, T, C_l, H_l, W_l)
                   where H_l = H / 2^l
        """
        B, T, C, H, W = input_seq.shape
        device = input_seq.device

        # Initialise states
        E_seq, R_seq, H_seq = [], [], []
        h, w = H, W
        for l in range(self.n_layers):
            E_seq.append(torch.zeros(B, 2 * self.a_channels[l], h, w, device=device))
            R_seq.append(torch.zeros(B, self.r_channels[l],      h, w, device=device))
            H_seq.append(None)
            h //= 2; w //= 2

        all_R = [[] for _ in range(self.n_layers)]  # [layer][t] -> (B, C_l, H_l, W_l)

        for t in range(T):
            A = input_seq[:, t].float()

            # Top-down LSTM update (reversed layers)
            for l in reversed(range(self.n_layers)):
                cell = getattr(self, f'cell{l}')
                hx = (R_seq[l], R_seq[l]) if t == 0 else H_seq[l]
                if l == self.n_layers - 1:
                    R, hx = cell(E_seq[l], hx)
                else:
                    tmp = torch.cat([E_seq[l], self.upsample(R_seq[l + 1])], 1)
                    R, hx = cell(tmp, hx)
                R_seq[l] = R
                H_seq[l] = hx

            # Capture R at this timestep
            for l in range(self.n_layers):
                all_R[l].append(R_seq[l].detach().cpu())

            # Bottom-up prediction error
            for l in range(self.n_layers):
                conv = getattr(self, f'conv{l}')
                A_hat = conv(R_seq[l])
                pos = F.relu(A_hat - A)
                neg = F.relu(A - A_hat)
                E = torch.cat([pos, neg], 1)
                E_seq[l] = E
                if l < self.n_layers - 1:
                    A = getattr(self, f'update_A{l}')(E)

        # Stack per layer: (B, T, C_l, H_l, W_l)
        return [torch.stack(all_R[l], dim=1) for l in range(self.n_layers)]

In [8]:
import numpy as np
import torch.nn.functional as F
import sys, os
_d = os.path.abspath(os.getcwd())
sys.path.insert(0, _d if os.path.isdir(os.path.join(_d, 'src')) else os.path.dirname(_d))
from src.plot_utils import createFlowDataset
from time import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

######################## PARAMS ########################
# PredNet architecture (matches original paper / KITTI training)
A_channels = (3, 48, 96, 192)
R_channels = (3, 48, 96, 192)
N_LAYERS    = len(R_channels)  # 4
PREDNET_SIZE = 128  # spatial input (must be divisible by 2^N_LAYERS = 16)

n_fmaps_to_sample = 40
samples_per_fmap  = 50
seed              = 17
N_INSTANCES       = 1
trial_len         = 37
NDIRS             = 8
scl_factor        = 0.7

# ImageNet normalisation (grayscale → RGB, same convention as nb09)
IMGNET_MEAN = np.array([0.485, 0.456, 0.406], dtype='float32')
IMGNET_STD  = np.array([0.229, 0.224, 0.225], dtype='float32')

# Optical-flow stimulus parameters (must match nb01)
topdir      = '../stimuli/flowstims'
orig_shape  = (800, 600)
input_shape = (128, 128)
mydirs      = ['0', '45', '90', '135', '180', '225', '270', '315']
categories  = [
    'grat_W12', 'grat_W1', 'grat_W2',
    'neg1dotflow_D1_bg', 'neg1dotflow_D2_bg',
    'neg3dotflow_D1_bg', 'neg3dotflow_D2_bg',
    'pos1dotflow_D1_bg', 'pos1dotflow_D2_bg',
    'pos3dotflow_D1_bg', 'pos3dotflow_D2_bg',
]
NSTIMS = len(categories)
assert NSTIMS == 11

n_orig_imgs    = NSTIMS * NDIRS
n_shifted_imgs = n_orig_imgs * trial_len
print(f'n_orig_imgs={n_orig_imgs}, n_shifted_imgs={n_shifted_imgs}')

Using device: cpu
n_orig_imgs=88, n_shifted_imgs=3256


In [9]:
########## INSTANTIATE MODEL ##########
# No publicly available PyTorch pretrained weights.
# Random initialisation still demonstrates ConvLSTM temporal dynamics.
# To use Keras pretrained weights (KITTI), see coxlab/prednet README for download,
# then manually convert layer weights via h5py.

model = PredNet(R_channels, A_channels)
model.eval()
model = model.to(device)

# Determine per-layer output shapes via a warm-up pass
dummy_seq = torch.zeros(1, trial_len, 3, PREDNET_SIZE, PREDNET_SIZE, device=device)
with torch.no_grad():
    layer_outs = model.forward_capture(dummy_seq)

layer_shapes = {}  # layer_idx -> (C_l, H_l, W_l)
for l, lo in enumerate(layer_outs):
    _, T_, C_l, H_l, W_l = lo.shape
    layer_shapes[l] = (C_l, H_l, W_l)
    print(f'  L{l}: R_channels={R_channels[l]}, shape=(T={T_}, C={C_l}, H={H_l}, W={W_l})')

  L0: R_channels=3, shape=(T=37, C=3, H=128, W=128)
  L1: R_channels=48, shape=(T=37, C=48, H=64, W=64)
  L2: R_channels=96, shape=(T=37, C=96, H=32, W=32)
  L3: R_channels=192, shape=(T=37, C=192, H=16, W=16)


In [10]:
########## LOAD OPTICAL-FLOW STIMULI ##########

flow_datasets = createFlowDataset(
    categories, topdir, mydirs,
    orig_shape=orig_shape, input_shape=input_shape,
    scl_factor=scl_factor, N_INSTANCES=N_INSTANCES,
    trial_len=trial_len, stride=1,
)

for insti, arr in flow_datasets.items():
    print(f'Instance {insti}: shape={arr.shape}')
assert flow_datasets[0].shape[0] == n_shifted_imgs

*INSTANCE 0 ...........
Instance 0: shape=(3256, 16384)


In [11]:
########## PREPROCESSING HELPER ##########

def preprocess_clip_prednet(flat_trial, orig_h, orig_w, target_size):
    """Convert one 37-frame trial (T, H*W) uint8 → normalised RGB sequence tensor.

    Returns:
        torch.Tensor of shape (1, T, 3, target_size, target_size)
    """
    T = flat_trial.shape[0]
    frames = flat_trial.reshape(T, orig_h, orig_w).astype('float32') / 255.0  # (T, H, W)
    frames_rgb = np.stack([frames, frames, frames], axis=1)                    # (T, 3, H, W)
    frames_norm = (frames_rgb - IMGNET_MEAN[:, None, None]) / IMGNET_STD[:, None, None]
    t = torch.tensor(frames_norm)  # (T, 3, H, W)
    # Resize spatial dims: treat T as batch for interpolation
    t = F.interpolate(t, size=(target_size, target_size), mode='bilinear', align_corners=False)
    return t.unsqueeze(0)  # (1, T, 3, target_size, target_size)

In [12]:
########## FORWARD PASSES (all layers captured per-frame) ##########

orig_h, orig_w = input_shape  # 144, 256

# Accumulate R activations; shape per layer: (n_orig_imgs, C_l, H_l, W_l, trial_len)
# We store as (n_orig_imgs, trial_len, C_l, H_l, W_l) then transpose later
layer_outputs = {
    l: np.zeros((n_orig_imgs, trial_len, *layer_shapes[l]), dtype='float32')
    for l in range(N_LAYERS)
}

for insti in range(N_INSTANCES):
    extX = flow_datasets[insti]  # (n_shifted_imgs, H*W)
    extX_clips = extX.reshape(n_orig_imgs, trial_len, -1)  # (88, 37, H*W)
    print(f'Instance {insti}', flush=True)
    t0 = time()

    for img_idx in range(n_orig_imgs):
        clip_flat = extX_clips[img_idx]  # (37, H*W)
        clip_t = preprocess_clip_prednet(clip_flat, orig_h, orig_w, PREDNET_SIZE).to(device)
        # clip_t: (1, 37, 3, 128, 128)

        with torch.no_grad():
            r_list = model.forward_capture(clip_t)
        # r_list[l]: (1, T, C_l, H_l, W_l)

        for l in range(N_LAYERS):
            layer_outputs[l][img_idx] += r_list[l].squeeze(0).numpy()  # (T, C_l, H_l, W_l)

    print(f'  done in {time()-t0:.1f}s', flush=True)

for l in range(N_LAYERS):
    layer_outputs[l] /= N_INSTANCES
    print(f'L{l}: {layer_outputs[l].shape}  '
          f'min={layer_outputs[l].min():.4f}  max={layer_outputs[l].max():.4f}')

Instance 0
  done in 85.7s
L0: (88, 37, 3, 128, 128)  min=-1.0000  max=1.0000
L1: (88, 37, 48, 64, 64)  min=-1.0000  max=1.0000
L2: (88, 37, 96, 32, 32)  min=-1.0000  max=1.0000
L3: (88, 37, 192, 16, 16)  min=-1.0000  max=1.0000


In [13]:
########## SAMPLE NEURONS + BUILD TENSOR4D + SAVE (per layer) ##########

os.makedirs('../data/sampled', exist_ok=True)

for l in range(N_LAYERS):
    print(f'\n=== L{l} ===')
    C_l, H_l, W_l = layer_shapes[l]
    lo = layer_outputs[l]  # (n_orig_imgs, trial_len, C_l, H_l, W_l)

    # Reshape to (n_orig_imgs, C_l, H_l*W_l, trial_len) — consistent with nb09
    lo_t = lo.transpose(0, 2, 3, 4, 1)          # (n_orig_imgs, C_l, H_l, W_l, trial_len)
    lo_t = lo_t.reshape(n_orig_imgs, C_l, H_l * W_l, trial_len)
    n_neurons_per_fmap = H_l * W_l

    all_neurons_maxs  = lo_t.max(axis=(0, 3))   # (C_l, H_l*W_l)
    all_neurons_means = lo_t.mean(axis=(0, 3))  # (C_l, H_l*W_l)

    # -- Sample feature maps (maxFr) --
    np.random.seed(seed)
    nfmaps = C_l
    maxsmean = all_neurons_maxs.mean(1)                  # (C_l,)
    # For random-init model, activations may all be near zero at L0 (only 3 channels);
    # fall back to uniform sampling if maxsmean is near-zero
    if maxsmean.sum() < 1e-8:
        print(f'  WARNING: near-zero activations at L{l}, using uniform fmap sampling')
        n_fmaps_ = min(n_fmaps_to_sample, nfmaps)
        top_fmaps = np.random.choice(nfmaps, n_fmaps_, replace=False)
    else:
        nonzero_fmaps = int((~np.isclose(maxsmean, 0)).sum())
        n_fmaps_ = min(n_fmaps_to_sample, nonzero_fmaps, nfmaps)
        probs_fmap = maxsmean / maxsmean.sum()
        top_fmaps = np.random.choice(nfmaps, n_fmaps_, replace=False, p=probs_fmap)

    # -- Sample spatial positions within each fmap (maxNr) --
    samps_per_fmap = min(samples_per_fmap, n_neurons_per_fmap)
    sampled_neurons = []
    for fi in top_fmaps:
        neuron_vals = all_neurons_maxs[fi]
        if neuron_vals.sum() < 1e-8:
            top_nis = np.random.choice(n_neurons_per_fmap, samps_per_fmap, replace=False)
        else:
            nonzero_n = int((~np.isclose(neuron_vals, 0)).sum())
            samps     = min(samps_per_fmap, nonzero_n)
            probs_n   = neuron_vals / neuron_vals.sum()
            top_nis   = np.random.choice(n_neurons_per_fmap, samps, replace=False, p=probs_n)
        sampled_neurons += list(fi * n_neurons_per_fmap + top_nis)
    sampled_neurons = np.array(sampled_neurons)
    n_neurons_to_pick = len(sampled_neurons)
    print(f'  Sampled {n_neurons_to_pick} neurons from {n_fmaps_} feature maps')

    # -- Build tensor4d (N, NSTIMS, NDIRS, trial_len) --
    tensorX      = np.zeros((n_neurons_to_pick, NSTIMS, NDIRS, trial_len), dtype='float32')
    neurons_used = np.empty((n_neurons_to_pick, 3), dtype='int')  # (fmap_idx, h_pos, w_pos)

    for nii, ni in enumerate(sampled_neurons):
        fi   = ni // n_neurons_per_fmap
        posi = ni %  n_neurons_per_fmap
        hi   = posi // W_l
        wi   = posi %  W_l
        neurons_used[nii] = [fi, hi, wi]
        for cati in range(NSTIMS):
            pst = lo_t[cati * NDIRS:(cati + 1) * NDIRS, fi, posi, :]  # (NDIRS, trial_len)
            tensorX[nii, cati] = pst

    print(f'  tensorX shape: {tensorX.shape}')

    # -- Save --
    SUFFIX = f'prednet_L{l}_i{N_INSTANCES}_n{n_neurons_to_pick}_seed{seed}'
    out_tensor  = f'../data/sampled/tensor4d_{SUFFIX}.npy'
    out_neurons = f'../data/sampled/neurons_used_{SUFFIX}.npy'

    if os.path.exists(out_tensor):
        print(f'  [SKIP] {out_tensor} already exists — delete to regenerate')
    else:
        np.save(out_tensor, tensorX)
        np.save(out_neurons, neurons_used)
        print(f'  Saved {out_tensor}')
        print(f'  Saved {out_neurons}')


=== L0 ===
  Sampled 150 neurons from 3 feature maps
  tensorX shape: (150, 11, 8, 37)
  Saved ../data/sampled/tensor4d_prednet_L0_i1_n150_seed17.npy
  Saved ../data/sampled/neurons_used_prednet_L0_i1_n150_seed17.npy

=== L1 ===
  Sampled 2000 neurons from 40 feature maps
  tensorX shape: (2000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_prednet_L1_i1_n2000_seed17.npy
  Saved ../data/sampled/neurons_used_prednet_L1_i1_n2000_seed17.npy

=== L2 ===
  Sampled 2000 neurons from 40 feature maps
  tensorX shape: (2000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_prednet_L2_i1_n2000_seed17.npy
  Saved ../data/sampled/neurons_used_prednet_L2_i1_n2000_seed17.npy

=== L3 ===
  Sampled 2000 neurons from 40 feature maps
  tensorX shape: (2000, 11, 8, 37)
  Saved ../data/sampled/tensor4d_prednet_L3_i1_n2000_seed17.npy
  Saved ../data/sampled/neurons_used_prednet_L3_i1_n2000_seed17.npy
